# Анализ лояльности пользователей Яндекс Афиш

## Цели и задачи проекта:
С помощью датасета с информацией о клиентах и их активности, надо проанализировать и выявить портрет пользователя, который с большей вероятностью будет возвращаться на платформу и делать повторные заказы. Это поможет оптимизировать рекламные бюжеты и влиять на продолжительность удержания клиента

## Данные:
Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- user_id — уникальный идентификатор пользователя, совершившего заказ;
- device_type_canonical — тип устройства, с которого был оформлен заказ (mobile — мобильные устройства, desktop — стационарные);
- order_id — уникальный идентификатор заказа;
- order_dt — дата создания заказа (используйте данные created_dt_msk);
- order_ts — дата и время создания заказа (используйте данные created_ts_msk);
- currency_code — валюта оплаты;
- revenue — выручка от заказа;
- tickets_count — количество купленных билетов;
- days_since_prev — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- event_id — уникальный идентификатор мероприятия;
- service_name — название билетного оператора;
- event_type_main — основной тип мероприятия (театральная постановка, концерт и так далее);
- region_name — название региона, в котором прошло мероприятие;
- city_name — название города, в котором прошло мероприятие.


### 1. Загрузка данных и сбор общей информации

In [1]:
#Установка необходимых библиотек
!pip install phik


In [4]:
!pip install sqlalchemy

In [5]:
!pip install psycopg2

In [10]:
#Импорт необходимых библиотек для анализа данных
import pandas as pd
import numpy as np

#Импорт библиотек для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

#Импорт внутрненних библиотек для работы с чувствительными файлами
import os
import dotenv
from sqlalchemy import create_engine

In [12]:
#Подгружаю данные с доступом к БД через .env-файл
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'), # пароль
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'), # порт подключения
    'db': os.getenv('DB_NAME')
}

In [13]:
#Импортирую строку для подключения
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
) 

In [14]:
#Создаем соединение с БД
engine = create_engine(connection_string) 

In [15]:
#Проверяю подключение к БД
db_config

{'user': 'praktikum_student',
 'pwd': 'Sdf4$2;d-d30pp',
 'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
 'port': '6432',
 'db': 'data-analyst-afisha'}

In [16]:
#Выгружаем данные из БД с помощью SQL-запроса
query = '''
-- Настройка параметра synchronize_seqscans важна для проверки
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
-- Напишите ваш запрос ниже
SELECT p.user_id,
      p.device_type_canonical,
      p.order_id,
      p.created_dt_msk AS order_dt,
      p.created_ts_msk AS order_ts,
      p.currency_code,
      p.revenue,
      p.tickets_count,
      EXTRACT (DAY FROM (p.created_dt_msk - LAG(p.created_dt_msk) OVER (PARTITION BY p.user_id ORDER BY p.created_dt_msk)))::INTEGER AS days_since_prev,
      p.event_id,
      e.event_name_code AS event_name,
      e.event_type_main,
      p.service_name,
      r.region_name,
      c.city_name
FROM afisha.purchases p
JOIN afisha.events e USING(event_id)
JOIN afisha.city c USING(city_id)
JOIN afisha.regions r USING(region_id)
WHERE p.device_type_canonical IN ('mobile', 'desktop') AND e.event_type_main NOT IN ('фильм')
ORDER BY p.user_id;
'''

In [17]:
#Присваиваем то что извлекаем из БД переменной
df = pd.read_sql_query(query, con=engine)

In [19]:
#Проверяю результат выгрузки из БД
df.head(3)

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Край билетов,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Мой билет,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,За билетом!,Каменевский регион,Глиногорск


In [20]:
#Проверяю размер датасета
df.shape

(290611, 15)